**Table of contents**<a id='toc0_'></a>    
- 1. [生成式语言模型的对话模板介绍](#toc1_)    
- 2. [Lora微调后单独部署大模型输出结果不一致](#toc2_)    
  - 2.1. [异常原因](#toc2_1_)    
  - 2.2. [解决办法](#toc2_2_)    
    - 2.2.1. [思路](#toc2_2_1_)    
    - 2.2.2. [vllm聊天模板](#toc2_2_2_)    
    - 2.2.3. [LLaMAFactory聊天模板](#toc2_2_3_)    
    - 2.2.4. [模板转换脚本](#toc2_2_4_)    
    - 2.2.5. [模板转换脚本使用方法](#toc2_2_5_)    
    - 2.2.6. [jinjia转换结果：](#toc2_2_6_)    
    - 2.2.7. [转换结果的使用](#toc2_2_7_)    
    - 2.2.8. [运行多轮对话py脚本](#toc2_2_8_)    
    - 2.2.9. [退出对话](#toc2_2_9_)    
    - 2.2.10. [停止服务](#toc2_2_10_)    
- 3. [使用XTuner微调大模型](#toc3_)    
- 4. [如何导出LLama Factory的对话模板](#toc4_)    
- 5. [vllm推理模型时自定义对话模板](#toc5_)    
- 6. [案例：使用vllm有效部署Lora微调后的Qwen模型](#toc6_)    

<!-- vscode-jupyter-toc-config
	numbering=true
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

# 1. <a id='toc1_'></a>[生成式语言模型的对话模板介绍](#toc0_)

在微调本地数据集的时候，模型名称的作用就是让对话模板可以根据模型名称自动匹配，这里的对话模板并不是官方的对话模板，而是参考官方的对话模板做的自己的对话模板。

![](Image/2025-04-05-16-45-46.png)

对话模板控制着输出内容的格式和信息，不同模型的对话模板是不一样的，但是LLaMAFactory里面同一个系列的模型一般是一样的，没有进行细分，例如Qwen/Qwen2.5-1.5B-Instruct和Qwen/Qwen2.5-0.5B-Instruct的对话模板是同一个，都是Qwen。但是Vllm推理框架没有自己的模板，用的是模型自带的模板，1.5B和2.5B模型自带的模板是不一样的，导致训练时候和部署时候使用的对话模板不一致，就会导致回答内容有很大差异。

LLaMAFactory：对话模版是框架自己定义的，在设计的时候参考了模型自带的对话模板

Vllm：一般用模型自带的对话模板

open weibui：对话模版是框架自己定义的

![](Image/2025-04-05-16-59-00.png)

# 2. <a id='toc2_'></a>[Lora微调后单独部署大模型输出结果不一致](#toc0_)

## 2.1. <a id='toc2_1_'></a>[异常原因](#toc0_)

不同的模型部署工具“对话模板”可能是不一样的，例如LLaMAFactory的对话模板和vllm、Ollama的对话模板不一样。模型使用过程一般遇到三个框架，一个是微调(训练)框架、模型推理框架、前端框架（例如OpenWebUI），这三个过程可能用的模板都不一样，导致微调测试和前端使用输出结果大相径庭

## 2.2. <a id='toc2_2_'></a>[解决办法](#toc0_)

### 2.2.1. <a id='toc2_2_1_'></a>[思路](#toc0_)

解决微调训练框架(LLaMAFactory)和推理框架（vllm）模板不一直问题即----模板对齐

思路：
将LLaMAFactory微调训练时候的对话模板转换成推理框架（vllm）的.jinja模板格式，vllm加载转换后的模板

注：  
OpenWebUI用的是自己的对话模板，且没有加载自定义模板的接口。即便我们加载模型的时候用了自定义模板，后面OpenWebUI也会将其覆盖掉。所以自己微调过的模型很可能在OpenWebUI中没有作用。

### 2.2.2. <a id='toc2_2_2_'></a>[vllm聊天模板](#toc0_)

[vllm聊天模板](https://docs.vllm.com.cn/en/latest/serving/openai_compatible_server.html#chat-template)

![](Image/2025-04-06-00-29-13.png)

vllm serve <model> --chat-template ./path-to-chat-template.jinja

### 2.2.3. <a id='toc2_2_3_'></a>[LLaMAFactory聊天模板](#toc0_)

模LLaMAFactory板所在位置

/root/LLaMA-Factory/src/llamafactory/data/template.py

![](Image/2025-04-06-00-32-44.png)

LLaMAFactory本身就带模板格式转换函数，但是是私有函数，外面无法调用，我们可以写个脚本放到template.py同级目录下，然后调用该函数

![](Image/2025-04-06-00-42-41.png)

上面这个函数是私有的，下面这个是public的，可以调用

![](Image/2025-04-06-00-53-57.png)

### 2.2.4. <a id='toc2_2_4_'></a>[模板转换脚本](#toc0_)

模板转换脚本mytest.py内容如下：

In [ ]:
# mytest.py
import sys
import os

# 将项目根目录添加到 Python 路径
root_dir = os.path.dirname(os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(__file__)))))
sys.path.append(root_dir)

from llamafactory.data.template import TEMPLATES
from transformers import AutoTokenizer

# 1. 初始化分词器（任意支持的分词器均可），目的是获取tokenizer对象
tokenizer = AutoTokenizer.from_pretrained("/root/AI-WSL/models/Qwen/Qwen2.5-1.5B-Instruct")

# 2. 获取模板对象
template_name = "qwen"  # 替换为你需要查看的模板名称，要和训练时的模板名称一致，llamafactory里面是"qwen"
template = TEMPLATES[template_name]

# 3. 修复分词器的 Jinja 模板
template.fix_jinja_template(tokenizer)

# 4. 直接输出模板的 Jinja 格式
print("=" * 40)
print(f"Template [{template_name}] 的 Jinja 格式:")
print("=" * 40)
print(tokenizer.chat_template)

![](Image/2025-04-06-00-49-53.png)

### 2.2.5. <a id='toc2_2_5_'></a>[模板转换脚本使用方法](#toc0_)

将mytest.py拷贝到template.py同级目录，例如：/root/ModelFineTuningTool/LLaMaFactory/src/llamafactory/data/mytest.py

<img src="./Image/2025-04-30-22-32-53.png" style="margin-left: 0" width="30%">


切换到llamafactroy环境，执行mytest.py

![](Image/2025-04-30-22-53-25.png)

将输出结果复制保存到文件中

<img src="./Image/2025-04-30-22-55-25.png" style="margin-left: 0" width="85%">

### 2.2.6. <a id='toc2_2_6_'></a>[jinjia转换结果：](#toc0_)

{%- if tools %}
    {{- '<|im_start|>system\n' }}
    {%- if messages[0]['role'] == 'system' %}
        {{- messages[0]['content'] }}
    {%- else %}
        {{- 'You are Qwen, created by Alibaba Cloud. You are a helpful assistant.' }}
    {%- endif %}
    {{- "\n\n# Tools\n\nYou may call one or more functions to assist with the user query.\n\nYou are provided with function signatures within <tools></tools> XML tags:\n<tools>" }}
    {%- for tool in tools %}
        {{- "\n" }}
        {{- tool | tojson }}
    {%- endfor %}
    {{- "\n</tools>\n\nFor each function call, return a json object with function name and arguments within <tool_call></tool_call> XML tags:\n<tool_call>\n{\"name\": <function-name>, \"arguments\": <args-json-object>}\n</tool_call><|im_end|>\n" }}
{%- else %}
    {%- if messages[0]['role'] == 'system' %}
        {{- '<|im_start|>system\n' + messages[0]['content'] + '<|im_end|>\n' }}
    {%- else %}
        {{- '<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n' }}
    {%- endif %}
{%- endif %}
{%- for message in messages %}
    {%- if (message.role == "user") or (message.role == "system" and not loop.first) or (message.role == "assistant" and not message.tool_calls) %}
        {{- '<|im_start|>' + message.role + '\n' + message.content + '<|im_end|>' + '\n' }}
    {%- elif message.role == "assistant" %}
        {{- '<|im_start|>' + message.role }}
        {%- if message.content %}
            {{- '\n' + message.content }}
        {%- endif %}
        {%- for tool_call in message.tool_calls %}
            {%- if tool_call.function is defined %}
                {%- set tool_call = tool_call.function %}
            {%- endif %}
            {{- '\n<tool_call>\n{"name": "' }}
            {{- tool_call.name }}
            {{- '", "arguments": ' }}
            {{- tool_call.arguments | tojson }}
            {{- '}\n</tool_call>' }}
        {%- endfor %}
        {{- '<|im_end|>\n' }}
    {%- elif message.role == "tool" %}
        {%- if (loop.index0 == 0) or (messages[loop.index0 - 1].role != "tool") %}
            {{- '<|im_start|>user' }}
        {%- endif %}
        {{- '\n<tool_response>\n' }}
        {{- message.content }}
        {{- '\n</tool_response>' }}
        {%- if loop.last or (messages[loop.index0 + 1].role != "tool") %}
            {{- '<|im_end|>\n' }}
        {%- endif %}
    {%- endif %}
{%- endfor %}
{%- if add_generation_prompt %}
    {{- '<|im_start|>assistant\n' }}
{%- endif %}

### 2.2.7. <a id='toc2_2_7_'></a>[转换结果的使用](#toc0_)

复制jinjia转换结果到文件保存到某个目录下

![](Image/2025-05-01-19-19-47.png)

启用vllm的时候进行加载

--普通启动vllm的方式加载模板：  
vllm serve <model> --chat-template ./path-to-chat-template.jinja



![](Image/2025-05-01-19-23-32.png)


vllm serve /root/AI-WSL/models/Qwen/Qwen2.5-1.5B-Instruct-merged --chat-template /root/AI-WSL/data/qwen.jinjia

--docker启动vllm的方式加载模板：

Dockerfile 的 CMD 指令中增加 --chat-template 参数指定模板路径


FROM nvcr.io/nvidia/tritonserver:25.03-vllm-python-py3
WORKDIR /app
RUN mkdir -p /app/templates  # 创建模板目录
EXPOSE 8000
CMD ["python3", "-m", "vllm.entrypoints.openai.api_server", \
    "--model", "/app/models/Qwen/Qwen2.5-1.5B-Instruct-merged", \
    "--port", "8000", "--max_model_len", "512", \
    "--tensor-parallel-size", "1", \
    "--gpu_memory_utilization", "0.85", \
    "--max_num_seqs", "128", \
    "--enforce-eager", \
    # 新增模板参数  
    "--chat-template", "/app/templates/qwen.jinja"]

挂载聊天模板文件到容器  
在 docker run 命令中新增 -v 参数，将宿主机模板文件映射到容器内指定路径：

docker run -d \
  --name vllm-server \
  -p 8000:8000 \
  -v /root/AI-WSL/models/Qwen/Qwen2.5-1.5B-Instruct-merged:/app/models/Qwen/Qwen2.5-1.5B-Instruct-merged \
  -v /root/AI-WSL/data/qwen.jinjia:/app/templates/qwen.jinja \  # 新增模板映射
  --gpus all \
  --shm-size 16G \
  vllm-image


构建镜像  
docker build -t vllm-imagejinjia .

![](Image/2025-05-01-20-05-15.png)

运行容器

docker run -d \
  --name vllmjinjia-server \
  -p 8000:8000 \
  -v /root/AI-WSL/models/Qwen/Qwen2.5-1.5B-Instruct-merged:/app/models/Qwen/Qwen2.5-1.5B-Instruct-merged \
  -v /root/AI-WSL/data/qwen.jinjia:/app/templates/qwen.jinja \
  --gpus all \
  --shm-size 16G \
  vllm-imagejinjia

运行后Docker Desktop会出现该容器

![](Image/2025-05-01-23-57-58.png)

检查容器运行状态

docker ps -a | grep "vllmjinjia-server"

正常状态‌：STATUS 显示为 Up，且端口映射正确
异常处理‌：若状态为 Exited，使用 docker logs vllm-server 查看错误日志（如模型路径错误、显存不足等）

![](Image/2025-05-02-00-13-45.png)

启动容器-docker start  
第一次使用docker run以后就不能再用了  
docker start vllmjinjia-server

### 2.2.8. <a id='toc2_2_8_'></a>[运行多轮对话py脚本](#toc0_)

先进入.py文件所在目录  
cd /root/AI-WSL/project/dockervllmjinjia  
python test03.py

![](Image/2025-05-02-00-18-53.png)

### 2.2.9. <a id='toc2_2_9_'></a>[退出对话](#toc0_)
Ctrl+c

### 2.2.10. <a id='toc2_2_10_'></a>[停止服务](#toc0_)
docker stop vllmjinjia-server

# 3. <a id='toc3_'></a>[使用XTuner微调大模型](#toc0_)

XTuner和LLama Factory使用方式上完全不同，LLaMA Factory是以可视化界面操作为主主流的框架XTuner恰恰相反，主要通过命令行和文件配置来微调模型，模型测试的方式也不一样，XTuner训练过程中以主观评价为主，LLaMA Factory在训练过程中是无法执行主观评价测试的，训练完成后在chat界面执行主观测试。

# 4. <a id='toc4_'></a>[如何导出LLama Factory的对话模板](#toc0_)

# 5. <a id='toc5_'></a>[vllm推理模型时自定义对话模板](#toc0_)

# 6. <a id='toc6_'></a>[案例：使用vllm有效部署Lora微调后的Qwen模型](#toc0_)